# Задача 4: замеры и обоснование выбора инструментов

Что измеряем для раздела «Программные средства» отчёта к РК-1:
- размер датасета в CSV и в Parquet;
- время чтения в pandas из CSV и из Parquet;
- время агрегации в DuckDB прямо по файлу;
- память машины.

Перед запуском выполните raw/raw.ipynb: он скачивает данные и делает Parquet. Если processes и features уже выполнены, их статистика тоже попадёт в презентацию. В конце ноутбук сохраняет results.json, zamery_table.csv и собирает presentation.pptx.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import importlib
import json
import os
import platform
import re
import statistics
import time
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
import psutil

CSV = Path('raw/building_violations.csv')
PARQUET = Path('raw/building_violations.parquet')
assert CSV.exists() and PARQUET.exists(), 'Сначала выполните raw/raw.ipynb'


def load_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


RAW_STATS = load_json('raw/raw_stats.json') or {}
CSV_OPTS = RAW_STATS.get('csv_opts', 'header=true, all_varchar=true')
PQ = f"'{PARQUET.as_posix()}'"
CSV_SRC = f"read_csv('{CSV.as_posix()}', {CSV_OPTS})"
con = duckdb.connect()


def snake(name):
    return re.sub(r'[^0-9a-z]+', '_', name.strip().lower()).strip('_')


def bench(fn, runs):
    times = []
    for _ in range(runs):
        gc.collect()
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), times


def ru(x, d=0):
    return f'{x:,.{d}f}'.replace(',', ' ').replace('.', ',')


def plural(n, one, few, many):
    n = abs(int(n)) % 100
    if 11 <= n <= 14:
        return many
    n %= 10
    return one if n == 1 else few if 2 <= n <= 4 else many

## 1. Машина

In [3]:
vm = psutil.virtual_memory()
machine = {
    'ram_total_gb': round(vm.total / 1e9, 1),
    'ram_available_gb': round(vm.available / 1e9, 1),
    'cpu': os.cpu_count(),
    'os': platform.platform(),
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'duckdb': duckdb.__version__,
}
machine

{'ram_total_gb': 8.4,
 'ram_available_gb': 7.6,
 'cpu': 2,
 'os': 'Linux-6.18.44-fc-v37-x86_64-with-glibc2.39',
 'python': '3.11.15',
 'pandas': '3.0.6',
 'duckdb': '1.5.5'}

## 2. Размер файлов

In [4]:
csv_mb = CSV.stat().st_size / 1e6
parquet_mb = PARQUET.stat().st_size / 1e6
print(f'CSV:     {csv_mb:,.1f} МБ')
print(f'Parquet: {parquet_mb:,.1f} МБ, в {csv_mb / parquet_mb:.1f} раза меньше')

CSV:     930.4 МБ
Parquet: 149.3 МБ, в 6.2 раза меньше


## 3. Поместится ли таблица в pandas

pandas держит всю таблицу в оперативной памяти, а строки там занимают намного больше места, чем в CSV. Оцениваем объём по выборке из 200 000 строк. Если все столбцы не помещаются с запасом, замеры pandas делаем на 12 рабочих столбцах проекта — и это само по себе аргумент для отчёта.

In [5]:
WORK_CSV = ['ID', 'VIOLATION DATE', 'VIOLATION CODE', 'VIOLATION STATUS', 'VIOLATION STATUS DATE',
            'VIOLATION DESCRIPTION', 'INSPECTOR ID', 'INSPECTION NUMBER', 'INSPECTION STATUS',
            'INSPECTION CATEGORY', 'DEPARTMENT BUREAU', 'PROPERTY GROUP']
WORK_PQ = [snake(c) for c in WORK_CSV]

n_rows = con.sql(f'SELECT count(*) FROM {PQ}').fetchone()[0]
sample = pd.read_csv(CSV, nrows=200_000, low_memory=False)
est_gb = sample.memory_usage(deep=True).sum() / len(sample) * n_rows / 1e9
avail_gb = psutil.virtual_memory().available / 1e9
FULL = est_gb < 0.6 * avail_gb
del sample
gc.collect()

print(f'строк: {n_rows:,}')
print(f'оценка памяти pandas для всех столбцов: {est_gb:.1f} ГБ, свободно: {avail_gb:.1f} ГБ')
print('Режим: все столбцы' if FULL else 'Режим: 12 рабочих столбцов, все не поместятся с запасом')

строк: 2,030,101
оценка памяти pandas для всех столбцов: 1.1 ГБ, свободно: 7.2 ГБ
Режим: все столбцы


## 4. Чтение в pandas: CSV против Parquet

CSV — медиана 3 прогонов (каждый прогон долгий), Parquet — медиана 5.

In [6]:
def read_csv():
    if FULL:
        return pd.read_csv(CSV, low_memory=False)
    return pd.read_csv(CSV, usecols=WORK_CSV, low_memory=False)


def read_parquet():
    if FULL:
        return pd.read_parquet(PARQUET)
    return pd.read_parquet(PARQUET, columns=WORK_PQ)


csv_s, csv_runs = bench(read_csv, runs=3)
print(f'pd.read_csv:     {csv_s:.2f} с', [round(t, 2) for t in csv_runs])
pq_s, pq_runs = bench(read_parquet, runs=5)
print(f'pd.read_parquet: {pq_s:.2f} с', [round(t, 2) for t in pq_runs])
print(f'Parquet читается быстрее в {csv_s / pq_s:.1f} раза')

pd.read_csv:     15.25 с [26.58, 15.25, 13.53]


pd.read_parquet: 1.58 с [2.41, 1.06, 0.89, 1.87, 1.58]
Parquet читается быстрее в 9.6 раза


In [7]:
df = read_parquet()
df_mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f'DataFrame в памяти: {df_mem_gb:.2f} ГБ ({df.shape[0]:,} строк × {df.shape[1]} столбцов)')

pd_group_s, _ = bench(lambda: df.groupby('inspection_status').size(), runs=15)
print(f'pandas groupby по уже загруженному DataFrame: {pd_group_s * 1000:.1f} мс')
del df
gc.collect()

DataFrame в памяти: 1.18 ГБ (2,030,101 строк × 26 столбцов)


pandas groupby по уже загруженному DataFrame: 46.6 мс


0

## 5. Агрегация в DuckDB

Тот же вопрос, что и в pandas: сколько записей с каждым результатом инспекции. DuckDB читает из Parquet только нужный столбец и не загружает таблицу в память. Медиана 15 прогонов.

In [8]:
Q1 = f'SELECT inspection_status, count(*) AS n FROM {PQ} GROUP BY 1 ORDER BY n DESC'
duck_pq_s, _ = bench(lambda: con.sql(Q1).fetchall(), runs=15)

Q1_CSV = f'SELECT "INSPECTION STATUS" AS inspection_status, count(*) AS n FROM {CSV_SRC} GROUP BY 1'
duck_csv_s, _ = bench(lambda: con.sql(Q1_CSV).fetchall(), runs=3)

print(f'DuckDB по Parquet: {duck_pq_s * 1000:.1f} мс')
print(f'DuckDB по CSV:     {duck_csv_s:.2f} с')
con.sql(Q1).df()

DuckDB по Parquet: 11.6 мс
DuckDB по CSV:     0.98 с


,inspection_status,n
0,FAILED,1385395
1,PASSED,389018
2,CLOSED,255502
3,HOLD,170
4,NaN,16


## 6. Запрос ближе к реальному отчёту

Доля проваленных проверок по годам и службам. pandas: прочитать нужные столбцы и сгруппировать. DuckDB: один SQL-запрос по файлу.

In [9]:
Q2 = f"""
SELECT year(violation_date) AS year, department_bureau,
       count(*) AS n,
       avg(CASE WHEN inspection_status = 'FAILED' THEN 1 ELSE 0 END) AS fail_share
FROM {PQ}
GROUP BY 1, 2
ORDER BY 1, 3 DESC
"""


def q2_pandas():
    d = pd.read_parquet(PARQUET, columns=['violation_date', 'department_bureau', 'inspection_status'])
    failed = d['inspection_status'].eq('FAILED')
    return failed.groupby([d['violation_date'].dt.year, d['department_bureau']]).agg(['size', 'mean'])


q2_pd_s, _ = bench(q2_pandas, runs=3)
q2_duck_s, _ = bench(lambda: con.sql(Q2).fetchall(), runs=5)
print(f'pandas, чтение + groupby: {q2_pd_s:.2f} с')
print(f'DuckDB, SQL по файлу:     {q2_duck_s:.3f} с, быстрее в {q2_pd_s / q2_duck_s:.1f} раза')
con.sql(Q2).df().tail(10)

pandas, чтение + groupby: 0.32 с
DuckDB, SQL по файлу:     0.041 с, быстрее в 7.8 раза


,year,department_bureau,n,fail_share
288,2026,BOILER,1423,0.000000
289,2026,PLUMBING,1177,0.666950
290,2026,ELECTRICAL,623,0.996790
291,2026,SPECIAL INSPECTION PROGRAM,604,0.960265
292,2026,NEW CONSTRUCTION,499,0.977956
293,2026,REFRIGERATION,421,0.876485
294,2026,VENTILATION,372,0.548387
295,2026,SPECIAL TASK FORCE,63,1.000000
296,2026,DEMOLITION,48,0.000000
297,2026,SIGNS,44,0.954545


## 7. Профиль датасета для слайдов

In [10]:
prof = con.sql(f"""
SELECT count(*) AS rows,
       min(violation_date) AS date_from,
       max(violation_date) AS date_to,
       count(DISTINCT property_group) AS objects,
       count(DISTINCT violation_code) AS violation_types,
       count(DISTINCT inspection_number) AS inspections
FROM {PQ}
""").df().iloc[0]

status = con.sql(f"""
SELECT coalesce(nullif(upper(trim(inspection_status)), ''), 'UNKNOWN') AS status, count(*) AS n
FROM {PQ}
GROUP BY 1
ORDER BY n DESC
""").df()
status['share'] = status['n'] / status['n'].sum()
n_cols = len(con.sql(f'DESCRIBE SELECT * FROM {PQ}').df())
print(prof)
status

rows                           2030101
date_from          2006-01-01 00:00:00
date_to            2026-09-25 00:00:00
objects                         155704
violation_types                   1574
inspections                     470061
Name: 0, dtype: object


,status,n,share
0,FAILED,1385395,0.682427
1,PASSED,389018,0.191625
2,CLOSED,255502,0.125857
3,HOLD,170,0.000084
4,UNKNOWN,16,0.000008


## 8. Итоговая таблица для отчёта

In [11]:
table = pd.DataFrame([
    ['Размер исходного файла', 'CSV', f'{ru(csv_mb, 2)} МБ'],
    ['Размер сжатого файла', 'Parquet (Snappy)', f'{ru(parquet_mb, 2)} МБ'],
    ['Время чтения в память', 'pandas, CSV', f'{ru(csv_s, 3)} с'],
    ['Время чтения из файла', 'pandas, Parquet', f'{ru(pq_s, 3)} с'],
    ['Агрегация по целевому столбцу', 'DuckDB SQL, Parquet', f'{ru(duck_pq_s, 4)} с'],
    ['Память машины', 'RAM всего / свободно',
     f"{ru(machine['ram_total_gb'], 1)} / {ru(machine['ram_available_gb'], 1)} ГБ"],
], columns=['Параметр замера', 'Формат / инструмент', 'Значение'])
table.to_csv('zamery_table.csv', index=False, encoding='utf-8-sig')
table

,Параметр замера,Формат / инструмент,Значение
0,Размер исходного файла,CSV,"930,39 МБ"
1,Размер сжатого файла,Parquet (Snappy),"149,29 МБ"
2,Время чтения в память,"pandas, CSV","15,250 с"
3,Время чтения из файла,"pandas, Parquet","1,581 с"
4,Агрегация по целевому столбцу,"DuckDB SQL, Parquet","0,0116 с"
5,Память машины,RAM всего / свободно,"8,4 / 7,6 ГБ"


## 9. Обоснование: абзац для отчёта

In [12]:
ratio = csv_mb / parquet_mb
speed = csv_s / pq_s
q2_speed = q2_pd_s / q2_duck_s

if FULL:
    memory_part = (f'полная таблица занимает в pandas {ru(df_mem_gb, 1)} ГБ '
                   f'при {ru(avail_gb, 1)} ГБ свободной памяти, для предобработки pandas достаточно')
else:
    memory_part = (f'все столбцы заняли бы в pandas около {ru(est_gb, 1)} ГБ '
                   f'при {ru(avail_gb, 1)} ГБ свободной памяти, поэтому в pandas загружаются только рабочие столбцы')
if q2_speed >= 1.2:
    q2_part = f'сводный запрос по годам и службам выполняется в {ru(q2_speed, 1)} раза быстрее, чем в pandas'
else:
    q2_part = 'сводный запрос по годам и службам выполняется сопоставимо с pandas'

verdict = (
    f'Датасет занимает {ru(csv_mb / 1000, 2)} ГБ в CSV ({ru(n_rows)} {plural(n_rows, "запись", "записи", "записей")}). '
    f'В Parquet тот же набор весит {ru(parquet_mb)} МБ, в {ru(ratio, 1)} раза меньше, '
    f'и читается в pandas за {ru(pq_s, 1)} с против {ru(csv_s, 1)} с у CSV. '
    f'По памяти: {memory_part}. '
    f'Агрегация в DuckDB прямо по Parquet занимает {ru(duck_pq_s * 1000, 1)} мс, {q2_part}. '
    f"Данные помещаются на одну машину с {ru(machine['ram_total_gb'], 1)} ГБ памяти, обработка разовая, "
    'поэтому Spark избыточен: Parquet — для хранения, DuckDB — для агрегаций, pandas — для предобработки.'
)
print(verdict)

Датасет занимает 0,93 ГБ в CSV (2 030 101 запись). В Parquet тот же набор весит 149 МБ, в 6,2 раза меньше, и читается в pandas за 1,6 с против 15,2 с у CSV. По памяти: полная таблица занимает в pandas 1,2 ГБ при 7,2 ГБ свободной памяти, для предобработки pandas достаточно. Агрегация в DuckDB прямо по Parquet занимает 11,6 мс, сводный запрос по годам и службам выполняется в 7,8 раза быстрее, чем в pandas. Данные помещаются на одну машину с 8,4 ГБ памяти, обработка разовая, поэтому Spark избыточен: Parquet — для хранения, DuckDB — для агрегаций, pandas — для предобработки.


## 10. results.json и presentation.pptx

In [13]:
results = {
    'generated_at': datetime.now().strftime('%d.%m.%Y %H:%M'),
    'project': 'Автоматизация строительного контроля и управления качеством СМР',
    'team': 'Медиханов Тимур',
    'dataset': {
        'name': 'Chicago Building Violations',
        'rows': int(prof['rows']),
        'cols': int(n_cols),
        'date_from': pd.Timestamp(prof['date_from']).strftime('%d.%m.%Y'),
        'date_to': pd.Timestamp(prof['date_to']).strftime('%d.%m.%Y'),
        'objects': int(prof['objects']),
        'violation_types': int(prof['violation_types']),
        'inspections': int(prof['inspections']),
        'status_share': {r.status: round(float(r.share), 4) for r in status.itertuples()},
    },
    'sizes': {'csv_mb': round(csv_mb, 2), 'parquet_mb': round(parquet_mb, 2), 'ratio': round(ratio, 2)},
    'pandas': {
        'mode': 'full' if FULL else 'columns',
        'csv_read_s': round(csv_s, 3),
        'parquet_read_s': round(pq_s, 3),
        'speedup': round(speed, 2),
        'csv_runs': [round(t, 3) for t in csv_runs],
        'parquet_runs': [round(t, 3) for t in pq_runs],
        'est_gb': round(est_gb, 2),
        'avail_gb': round(avail_gb, 2),
        'df_mem_gb': round(df_mem_gb, 2),
        'groupby_s': round(pd_group_s, 5),
    },
    'duckdb': {
        'agg_parquet_s': round(duck_pq_s, 5),
        'agg_csv_s': round(duck_csv_s, 3),
        'q2_pandas_s': round(q2_pd_s, 3),
        'q2_duckdb_s': round(q2_duck_s, 4),
    },
    'machine': machine,
    'verdict': verdict,
    'raw': RAW_STATS or None,
    'prep': load_json('processes/prep_stats.json'),
    'features': load_json('features/feature_stats.json'),
}
Path('results.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('results.json сохранен')
if results['prep'] is None or results['features'] is None:
    print('Статистики processes/features нет: запустите их ноутбуки, затем эту ячейку и следующую ещё раз')

results.json сохранен


In [14]:
import make_presentation

importlib.reload(make_presentation)
make_presentation.build(results, 'presentation.pptx')

Презентация сохранена: /home/claude/stroykontrol_smr/presentation.pptx


'presentation.pptx'